# Importing Libraries


In [1]:
! pip install ucimlrepo

In [2]:
from ucimlrepo import fetch_ucirepo

# fetch dataset
heart_disease = fetch_ucirepo(id=45)

# data (as pandas dataframes)
X = heart_disease.data.features
y = heart_disease.data.targets.squeeze()

# metadata
print(heart_disease.metadata)

# variable information
print(heart_disease.variables)


{'uci_id': 45, 'name': 'Heart Disease', 'repository_url': 'https://archive.ics.uci.edu/dataset/45/heart+disease', 'data_url': 'https://archive.ics.uci.edu/static/public/45/data.csv', 'abstract': '4 databases: Cleveland, Hungary, Switzerland, and the VA Long Beach', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 303, 'num_features': 13, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': ['Age', 'Sex'], 'target_col': ['num'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1989, 'last_updated': 'Fri Nov 03 2023', 'dataset_doi': '10.24432/C52P4X', 'creators': ['Andras Janosi', 'William Steinbrunn', 'Matthias Pfisterer', 'Robert Detrano'], 'intro_paper': {'ID': 231, 'type': 'NATIVE', 'title': 'International application of a new probability algorithm for the diagnosis of coronary artery disease.', 'authors': 'R. Detrano, A. Jánosi, W. Steinbrunn, M

In [3]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler ,OneHotEncoder, OrdinalEncoder

import joblib

# Loading Dataset

In [4]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [5]:
X.isnull().sum()

,0
age,0
sex,0
cp,0
trestbps,0
chol,0
fbs,0
restecg,0
thalach,0
exang,0
oldpeak,0


In [6]:
X.duplicated().sum()

np.int64(0)

In [7]:
# Get summary statistics for categorical columns
X.describe().T

,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


In [8]:
for col in X:
    print(f"\nColumn: {col}")
    print("**"*20)
    print(X[col].value_counts())
    print("--"*20, '\n')


Column: age
****************************************
age
58    19
57    17
54    16
59    14
52    13
51    12
60    12
62    11
56    11
44    11
41    10
64    10
63     9
67     9
53     8
61     8
43     8
45     8
55     8
65     8
42     8
46     7
66     7
48     7
50     7
49     5
47     5
70     4
39     4
68     4
35     4
40     3
69     3
71     3
37     2
34     2
38     2
29     1
77     1
74     1
76     1
Name: count, dtype: int64
---------------------------------------- 


Column: sex
****************************************
sex
1    206
0     97
Name: count, dtype: int64
---------------------------------------- 


Column: cp
****************************************
cp
4    144
3     86
2     50
1     23
Name: count, dtype: int64
---------------------------------------- 


Column: trestbps
****************************************
trestbps
120    37
130    36
140    32
110    19
150    17
128    12
138    12
125    11
160    11
112     9
132     8
118     7
124     6


In [9]:
X.dtypes

,0
age,int64
sex,int64
cp,int64
trestbps,int64
chol,int64
fbs,int64
restecg,int64
thalach,int64
exang,int64
oldpeak,float64


In [10]:
# Specify categorical columns (numeric but categorical)
categorical_cols = [
    "sex",        # 0=female, 1=male
    "cp",         # chest pain type (0-3)
    "fbs",        # fasting blood sugar (0/1)
    "restecg",    # resting ECG (0-2)
    "exang",      # exercise induced angina (0/1)
    "slope",      # slope of ST segment (0-2)
    "ca",         # major vessels (0-3)
    "thal"        # thalassemia (3,6,7)
]
numerical_cols = [col for col in X.columns if col not in categorical_cols]


In [11]:
# Imputing for future missing input data
num_imputer = SimpleImputer(strategy='mean')
X[numerical_cols] = num_imputer.fit_transform(X[numerical_cols])
cat_imputer = SimpleImputer(strategy='most_frequent')
X[categorical_cols] = cat_imputer.fit_transform(X[categorical_cols])

In [12]:
# One-hot encoding for nominal categorical data
ordinal_cols = ["slope", "restecg", "ca"]      # has order
nominal_cols = ["sex", "cp", "fbs", "exang", "thal"]   # no order
# Ordinal encoding for ordinal categorical data
ord_encoder = OrdinalEncoder()
X[ordinal_cols] = ord_encoder.fit_transform(X[ordinal_cols])

# One-hot encoding for nominal categorical data
ohe = OneHotEncoder(sparse_output=False, drop="first")
nominal_encoded = ohe.fit_transform(X[nominal_cols])
nominal_encoded_df = pd.DataFrame(
    nominal_encoded,
    columns=ohe.get_feature_names_out(nominal_cols),
    index=X.index
)

# Drop nominal & concat encoded
X = X.drop(columns=nominal_cols)
X = pd.concat([X, nominal_encoded_df], axis=1)


In [13]:
X.dtypes


,0
age,float64
trestbps,float64
chol,float64
restecg,float64
thalach,float64
oldpeak,float64
slope,float64
ca,float64
sex_1.0,float64
cp_2.0,float64


In [14]:
# Scale numerical features
scaler = StandardScaler()
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])

In [15]:
X.head()

,age,trestbps,chol,restecg,thalach,oldpeak,slope,ca,sex_1.0,cp_2.0,cp_3.0,cp_4.0,fbs_1.0,exang_1.0,thal_6.0,thal_7.0
0,0.948726,0.757525,-0.264900,2.0,0.017197,1.087338,2.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0
1,1.392002,1.611220,0.760415,2.0,-1.821905,0.397182,1.0,3.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
2,1.392002,-0.665300,-0.342283,2.0,-0.902354,1.346147,1.0,2.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0
3,-1.932564,-0.096170,0.063974,0.0,1.637359,2.122573,2.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
4,-1.489288,-0.096170,-0.825922,2.0,0.980537,0.310912,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [16]:
# change target 2 binary class
# Original target
print("Original target distribution (5 classes):")
print(y.value_counts().sort_index())

Original target distribution (5 classes):
num
0    164
1     55
2     36
3     35
4     13
Name: count, dtype: int64


In [17]:
# Convert to binary: 0 = no disease, 1 = disease (classes 1-4)
target_binary = (y > 0).astype(int)
# Verify new distribution
print("\nBinary target distribution:")
print(pd.Series(target_binary).value_counts().sort_index())
print(f"Binary classes: {np.unique(target_binary)}")


Binary target distribution:
num
0    164
1    139
Name: count, dtype: int64
Binary classes: [0 1]


In [18]:
target_binary.to_csv('/content/drive/MyDrive/datasets/Heart_Disease_Project/target_binary.csv',
                     index=False, header=["target"])



In [19]:
# Save preprocessing objects
joblib.dump(num_imputer, '/content/drive/MyDrive/datasets/Heart_Disease_Project/num_imputer.pkl')
joblib.dump(cat_imputer, '/content/drive/MyDrive/datasets/Heart_Disease_Project/cat_imputer.pkl')
joblib.dump(scaler, '/content/drive/MyDrive/datasets/Heart_Disease_Project/scaler.pkl')
joblib.dump(ord_encoder, '/content/drive/MyDrive/datasets/Heart_Disease_Project/ordinal_encoder.pkl')
joblib.dump(ohe, '/content/drive/MyDrive/datasets/Heart_Disease_Project/onehot_encoder.pkl')
joblib.dump(list(X.columns), "/content/drive/MyDrive/datasets/Heart_Disease_Project/feature_columns.pkl")


print("Data preprocessing completed and saved.")

Data preprocessing completed and saved.
